In [1]:
!pip install azure-storage-blob pyarrow

Завдання 1. Підключення

In [2]:
from azure.storage.blob import ContainerClient
from io import BytesIO
import pandas as pd
import os

# Джерело (тільки читання)
SOURCE_SAS = os.environ['SOURCE_TOKEN']
# Озеро (читання + запис)
LAKE_SAS = os.environ['LAKE_TOKEN']

source = ContainerClient.from_container_url(SOURCE_SAS)
lake   = ContainerClient.from_container_url(LAKE_SAS)

#Завдання 1.1
STUDENT = "tyshyk_viktoriia"   # власний префікс

In [3]:
# читання JSON із джерела
def read_json(name):
    raw = source.download_blob(name).readall()
    return pd.read_json(BytesIO(raw))

# читання Parquet зі свого префікса в озері
def read_parquet(path):
    raw = lake.download_blob(f"{STUDENT}/{path}").readall()
    return pd.read_parquet(BytesIO(raw))

# запис DataFrame у Parquet у свій префікс
def write_parquet(df, path):
    buf = BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    lake.upload_blob(name=f"{STUDENT}/{path}", data=buf.getvalue(), overwrite=True)

In [4]:
#Завдання 1.2
shops=read_json("shops.json")
print(len(shops))

products=read_json("products.json")
print(len(products))

5
12


Завдання 2. Шар Bronze — сирі дані як є

In [5]:
#Завдання 2.1, 2.2
parts = [read_json(f"orders_2026_0{m}.json") for m in (1, 2, 3)]
bronze_orders = pd.concat(parts, ignore_index=True)
print(len(bronze_orders))

write_parquet(bronze_orders, "bronze/orders.parquet")

980


In [6]:
#Завдання 2.3
write_parquet(shops, "bronze/shops.parquet")
write_parquet(products, "bronze/products.parquet")

In [7]:
#Завдання 2.4
for blob in lake.list_blobs(name_starts_with=f"{STUDENT}/"):
    print(blob.name, blob.size)

tyshyk_viktoriia/bronze/orders.parquet 18620
tyshyk_viktoriia/bronze/products.parquet 3609
tyshyk_viktoriia/bronze/readings/readings.parquet 5062
tyshyk_viktoriia/bronze/sensors/sensors.parquet 4105
tyshyk_viktoriia/bronze/shops.parquet 4014
tyshyk_viktoriia/bronze/stations/stations.parquet 4840
tyshyk_viktoriia/gold/avg_by_city_metric/avg_by_city_metric.parquet 2976
tyshyk_viktoriia/gold/daily_trend/daily_trend.parquet 2470
tyshyk_viktoriia/gold/dim_product.parquet 4241
tyshyk_viktoriia/gold/dim_shop.parquet 4600
tyshyk_viktoriia/gold/fact_daily_sales.parquet 5501
tyshyk_viktoriia/gold/fact_sales.parquet 13765
tyshyk_viktoriia/gold/pm25_exceedances/pm25_exceedances.parquet 2957
tyshyk_viktoriia/nbu_rates/nbu_rates_2026-09-01_2026-09-07.csv 346
tyshyk_viktoriia/nbu_rates/nbu_rates_2026-10-14.csv 69
tyshyk_viktoriia/silver/_quality_report.json 346
tyshyk_viktoriia/silver/orders.parquet 22915
tyshyk_viktoriia/silver/products.parquet 3609
tyshyk_viktoriia/silver/readings/year=2026/month=0

Завдання 3. Шар Silver — очищені дані

In [8]:
#Завдання 3.1
orders_bronze=read_parquet("bronze/orders.parquet")
shops_bronze=read_parquet("bronze/shops.parquet")
products_bronze=read_parquet("bronze/products.parquet")

print("Кількість повних дублів: ", orders_bronze.duplicated().sum())
print("Кількість унікальних shop_code: ", orders_bronze["shop_code"].nunique())
print("Кількість унікальних payment_type: ", orders_bronze["payment_type"].nunique())
print("Кількість порожніх значень qty: ", orders_bronze["qty"].isna().sum())

Кількість повних дублів:  20
Кількість унікальних shop_code:  10
Кількість унікальних payment_type:  4
Кількість порожніх значень qty:  36


In [9]:
orders_silver=orders_bronze.copy()

#Завдання 3.2
orders_silver=orders_silver.drop_duplicates()

#Завдання 3.3
orders_silver["shop_code"] = orders_silver["shop_code"].str.strip().str.upper()
print("Кількість унікальних shop_code: ", orders_silver["shop_code"].nunique())

#Завдання 3.4
orders_silver["payment_type"]=orders_silver["payment_type"].str.strip().str.capitalize()
print("Кількість унікальних payment_type: ", orders_silver["payment_type"].nunique())

#Завдання 3.5
orders_silver["qty"]=orders_silver["qty"].fillna(1).astype(int)
print("Кількість порожніх значень qty: ", orders_silver["qty"].isna().sum())

#Завдання 3.6
orders_silver["order_ts"] = pd.to_datetime(orders_silver["order_ts"])
orders_silver["order_date"]=orders_silver['order_ts'].dt.date

#Завдання 3.7
orders_silver["amount"]=orders_silver["qty"]*orders_silver["unit_price"]

Кількість унікальних shop_code:  5
Кількість унікальних payment_type:  2
Кількість порожніх значень qty:  0


In [10]:
#Завдання 3.8, 3.9
write_parquet(orders_silver, "silver/orders.parquet")
write_parquet(shops_bronze, "silver/shops.parquet")
write_parquet(products_bronze, "silver/products.parquet")

Завдання 4. Шар Gold — зоряна схема

In [11]:
orders_silver=read_parquet("silver/orders.parquet")
shops_silver=read_parquet("silver/shops.parquet")
products_silver=read_parquet("silver/products.parquet")

In [12]:
#Завдання 4.1
dim_shop = shops_silver.sort_values("shop_code").reset_index(drop=True)
dim_shop.insert(0, "shop_key", range(1, len(dim_shop) + 1))

display(dim_shop)

,shop_key,shop_code,shop_name,city,district,opened_date,seats
0,1,S01,Kava Hub,Kyiv,Podil,2019-04-15,24
1,2,S02,Zerno,Kyiv,Pechersk,2021-09-01,16
2,3,S03,Rynok Coffee,Lviv,Center,2018-06-20,30
3,4,S04,Bereg,Odesa,Arkadia,2022-05-10,40
4,5,S05,Steam,Kharkiv,Center,2023-02-01,12


In [13]:
#Завдання 4.2
dim_product=products_silver.sort_values("product_code").reset_index(drop=True)
dim_product.insert(0, "product_key", range(1, len(dim_product)+1))
display(dim_product)

,product_key,product_code,product_name,category,size,price
0,1,P01,Espresso,Coffee,S,45
1,2,P02,Americano,Coffee,M,55
2,3,P03,Cappuccino,Coffee,M,70
3,4,P04,Latte,Coffee,L,80
4,5,P05,Flat White,Coffee,M,75
5,6,P06,Green Tea,Tea,M,50
6,7,P07,Herbal Tea,Tea,M,50
7,8,P08,Croissant,Bakery,-,60
8,9,P09,Cinnamon Bun,Bakery,-,65
9,10,P10,Sandwich,Bakery,-,110


In [14]:
#Завдання 4.3
fact_sales=(
    orders_silver.merge(
        dim_shop[["shop_key", "shop_code"]], on="shop_code", how="left", validate="m:1").merge(
        dim_product[["product_key", "product_code"]], on="product_code", how="left", validate="m:1")
        [["order_id", "shop_key", "product_key", "order_date",
          "qty", "unit_price", "amount", "payment_type"]]
)
print(len(fact_sales), fact_sales[["shop_key", "product_key"]].isna().sum().to_dict())
display(fact_sales.head())

960 {'shop_key': 0, 'product_key': 0}


,order_id,shop_key,product_key,order_date,qty,unit_price,amount,payment_type
0,O500259,4,4,2026-01-27,2,80,160,Card
1,O500252,5,4,2026-01-03,2,80,160,Card
2,O500257,4,11,2026-01-30,1,95,95,Cash
3,O500127,1,10,2026-01-03,3,110,330,Card
4,O500225,3,4,2026-01-28,3,80,240,Cash


In [15]:
#Завдання 4.4
fact_daily_sales=(
    fact_sales.groupby(["shop_key", "order_date"], as_index=False).agg(
        orders_count=("order_id", "nunique"),
        items_sold=("qty", "sum"),
        revenue=("amount", "sum"))
)
print(len(fact_daily_sales))
display(fact_daily_sales.head())

396


,shop_key,order_date,orders_count,items_sold,revenue
0,1,2026-01-01,2,3,235
1,1,2026-01-02,1,1,95
2,1,2026-01-03,6,9,785
3,1,2026-01-04,2,3,225
4,1,2026-01-05,2,3,170


In [16]:
#Завдання 4.5
write_parquet(dim_shop, "gold/dim_shop.parquet")
write_parquet(dim_product, "gold/dim_product.parquet")
write_parquet(fact_sales, "gold/fact_sales.parquet")
write_parquet(fact_daily_sales, "gold/fact_daily_sales.parquet")

Завдання 5. Перевірка та аналітика

In [17]:
dim_shop = read_parquet("gold/dim_shop.parquet")
dim_product = read_parquet("gold/dim_product.parquet")
fact_sales = read_parquet("gold/fact_sales.parquet")
fact_daily_sales = read_parquet("gold/fact_daily_sales.parquet")

In [18]:
#Завдання 5.1
print("dim_shop:", len(dim_shop))
print("dim_product:", len(dim_product))
print("fact_sales:", len(fact_sales))
print("fact_daily_sales:", len(fact_daily_sales))

dim_shop: 5
dim_product: 12
fact_sales: 960
fact_daily_sales: 396


In [19]:
#Завдання 5.2
print(fact_sales["shop_key"].isin(dim_shop["shop_key"]).all())
print(fact_sales["product_key"].isin(dim_product["product_key"]).all())

True
True


In [20]:
#Завдання 5.3
print(fact_sales["amount"].sum() == fact_daily_sales["revenue"].sum())

True


In [21]:
#Завдання 5.4
revenue_by_category = (
    fact_sales
    .merge(dim_product[["product_key", "category"]], on="product_key", how="left")
    .groupby("category", as_index=False)["amount"].sum()
    .rename(columns={"amount": "виторг"})
    .sort_values("виторг", ascending=False)
    .reset_index(drop=True)
)
display(revenue_by_category)

,category,виторг
0,Coffee,49915
1,Bakery,19955
2,Dessert,14850
3,Tea,5600


In [22]:
#Завдання 5.5
revenue_by_shop = (
    fact_sales
    .merge(dim_shop[["shop_key", "shop_name"]], on="shop_key", how="left")
    .groupby("shop_name", as_index=False)["amount"].sum()
    .rename(columns={"amount": "виторг"})
    .sort_values("виторг", ascending=False)
    .reset_index(drop=True)
)
display(revenue_by_shop)

,shop_name,виторг
0,Rynok Coffee,20075
1,Kava Hub,19290
2,Steam,18275
3,Bereg,16750
4,Zerno,15930
